# MMD Base 1B — Kaggle GPU notebook
Start this notebook manually in Kaggle. Attach a dataset containing corpus.txt and provenance.json. No pretrained weights or external inference APIs are used.

In [ ]:
!git clone --depth 1 --branch feat/mmd-cloud-training-mct-02 https://github.com/pctaha501-create/MMD-GameForge.git /kaggle/working/MMD-GameForge
%pip -q install tokenizers bitsandbytes
import pathlib, shutil
REPO = pathlib.Path('/kaggle/working/MMD-GameForge')
DATA_DIR = pathlib.Path('/kaggle/input')
matches = list(DATA_DIR.rglob('corpus.txt'))
if len(matches) != 1: raise RuntimeError(f'Expected exactly one corpus.txt; found {len(matches)}')
DATASET = matches[0]
PROVENANCE = DATASET.with_name('provenance.json')
if not PROVENANCE.exists(): raise FileNotFoundError('Place provenance.json next to corpus.txt')
OUTPUT = pathlib.Path('/kaggle/working/mmd-cloud-run')
checkpoint_matches = list(DATA_DIR.rglob('checkpoint.pt'))
if len(checkpoint_matches) > 1: raise RuntimeError('Attach at most one previous checkpoint directory as a Kaggle input dataset.')
if checkpoint_matches:
    shutil.copytree(checkpoint_matches[0].parent, OUTPUT, dirs_exist_ok=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
if not torch.cuda.is_available(): raise RuntimeError('Enable a GPU accelerator in Kaggle settings before training.')
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU memory: {gpu_gib:.2f} GiB; required preflight: 24 GiB')
if gpu_gib < 24: raise RuntimeError('MMD Base 1B requires at least 24 GiB GPU memory for this preflight. No CPU or paid fallback.')


In [ ]:
!python /kaggle/working/MMD-GameForge/cloud/runtime/train.py --config /kaggle/working/MMD-GameForge/cloud/configs/mmd-base-1b-bilingual.json --dataset "$DATASET" --provenance "$PROVENANCE" --output "$OUTPUT"


In [ ]:
import json, pathlib, subprocess, sys, shutil
metrics = json.loads((OUTPUT / 'metrics.json').read_text(encoding='utf-8'))
if not metrics.get('training_complete'):
    resume_zip = shutil.make_archive('/kaggle/working/mmd-cloud-resume', 'zip', OUTPUT)
    print('Training is incomplete. Preserve/download this checkpoint archive, attach it as a Kaggle dataset, and rerun to resume:', resume_zip)
    raise RuntimeError(f"Training paused at step {metrics.get('global_step')}/{metrics.get('max_steps')}; no release bundle was created.")
EVALUATION = OUTPUT / 'evaluation.json'
CONFIG = REPO / 'cloud/configs/mmd-base-1b-bilingual.json'
subprocess.run([sys.executable, str(REPO / 'cloud/runtime/evaluate.py'), '--config', str(CONFIG), '--dataset', str(DATASET), '--provenance', str(PROVENANCE), '--checkpoint', str(OUTPUT / 'checkpoint.pt'), '--tokenizer', str(OUTPUT / 'tokenizer.json'), '--output', str(EVALUATION)], check=True)
BUNDLE = pathlib.Path('/kaggle/working/mmd-base-1b-bilingual-bundle.zip')
subprocess.run([sys.executable, str(REPO / 'cloud/runtime/bundle.py'), '--checkpoint-dir', str(OUTPUT), '--config', str(CONFIG), '--evaluation', str(EVALUATION), '--output', str(BUNDLE)], check=True)
print('Verified model bundle:', BUNDLE)
